# Unlimited-OCR — Demo trên Google Colab / Kaggle (GPU T4 free)

Notebook này chạy `baidu/Unlimited-OCR` (3.34B) theo đường **transformers** trên GPU T4 16GB,
demo vài trang đầu của một file PDF (mặc định: `Nihongo Sou Matome N1 Dokkai.pdf`).

**Cách dùng:**
1. Mở notebook này trên Colab → Menu **Runtime → Change runtime type → T4 GPU**.
2. Chạy lần lượt từng cell (Shift+Enter).
3. Ở cell **Upload PDF**, chọn file PDF từ máy của bạn.
4. Kết quả markdown của từng trang sẽ in ra và lưu trong thư mục `outputs/`.

> Lưu ý: chỉ chạy `NUM_PAGES` trang đầu (mặc định 3) cho nhanh. PDF đầy đủ ~130 trang sẽ rất lâu.

## 1. Kiểm tra GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv

## 2. Cài dependencies
Giữ nguyên `torch` có sẵn của Colab (đã khớp CUDA của runtime) để tránh cài lại nặng/lỗi.
Chỉ cài thêm các thư viện model cần.

In [ ]:
!pip install -q \
    transformers==4.57.1 \
    "Pillow>=11" \
    einops==0.8.2 \
    addict==2.4.0 \
    easydict==1.13 \
    pymupdf==1.27.2.2 \
    matplotlib \
    psutil
print('done')

## 3. Upload file PDF
Chạy cell rồi chọn file `Nihongo Sou Matome N1 Dokkai.pdf` (hoặc PDF bất kỳ).
Trên Kaggle thay bằng đường dẫn dataset, ví dụ `PDF_PATH = '/kaggle/input/.../file.pdf'`.

In [ ]:
try:
    from google.colab import files
    uploaded = files.upload()
    PDF_PATH = next(iter(uploaded))
except Exception:
    # Kaggle / local: chỉnh đường dẫn thủ công
    PDF_PATH = 'Nihongo Sou Matome N1 Dokkai.pdf'
print('PDF_PATH =', PDF_PATH)

## 4. Tải model (~7 GB, lần đầu mất vài phút)

In [ ]:
import torch
from transformers import AutoModel, AutoTokenizer

model_name = 'baidu/Unlimited-OCR'
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModel.from_pretrained(
    model_name,
    trust_remote_code=True,
    use_safetensors=True,
    torch_dtype=torch.bfloat16,
)
model = model.eval().cuda()
print('Model loaded:', sum(p.numel() for p in model.parameters()) / 1e9, 'B params')

## 5. Chuyển vài trang PDF -> ảnh

In [ ]:
import os, tempfile
import fitz  # PyMuPDF

NUM_PAGES = 3      # số trang demo (tăng nếu muốn, nhưng sẽ lâu hơn)
START_PAGE = 0     # bắt đầu từ trang nào (0-based)
DPI = 200          # 200-300; cao hơn = nét hơn nhưng nặng hơn

def pdf_to_images(pdf_path, start, count, dpi):
    doc = fitz.open(pdf_path)
    tmp_dir = tempfile.mkdtemp(prefix='pdf_ocr_')
    mat = fitz.Matrix(dpi / 72, dpi / 72)
    paths = []
    end = min(start + count, doc.page_count)
    for i in range(start, end):
        out = os.path.join(tmp_dir, f'page_{i+1:04d}.png')
        doc[i].get_pixmap(matrix=mat).save(out)
        paths.append(out)
    doc.close()
    return paths

image_files = pdf_to_images(PDF_PATH, START_PAGE, NUM_PAGES, DPI)
print(f'{len(image_files)} trang -> ảnh:')
for p in image_files:
    print(' ', p)

## 6. Chạy OCR đa trang (single forward pass)

In [ ]:
import time
os.makedirs('outputs', exist_ok=True)

t0 = time.time()
result = model.infer_multi(
    tokenizer,
    prompt='<image>Multi page parsing.',
    image_files=image_files,
    output_path='outputs',
    image_size=1024,
    max_length=32768,
    no_repeat_ngram_size=35,
    ngram_window=1024,
    save_results=True,
)
print(f'\n--- Xong trong {time.time()-t0:.1f}s ---')

## 7. Xem kết quả

In [ ]:
from IPython.display import Markdown, display
import glob

# In text trả về trực tiếp (nếu có)
if isinstance(result, str) and result.strip():
    display(Markdown(result))

# Và/hoặc đọc các file đã lưu trong outputs/
for f in sorted(glob.glob('outputs/**/*', recursive=True)):
    if os.path.isfile(f) and f.lower().endswith(('.md', '.txt')):
        print('=' * 60, '\n', f, '\n', '=' * 60)
        display(Markdown(open(f, encoding='utf-8').read()))